# 🎬 SeedVR2-cli — Google Colab 影像與影片超解析度放大

支援 **Tesla T4 (15GB)** 與 **NVIDIA L4 (22GB)** 等 GPU 執行階段。
無需複雜設定，依序執行下方 Cell 即可快速開始！


In [ ]:
# @title 1. 檢測 GPU 環境與掛載 Google Drive
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ 請先在 Colab 選單「執行階段」->「變更執行階段類型」選取 GPU (T4 或 L4)！")

props = torch.cuda.get_device_properties(0)
print(f"✅ 偵測到 GPU: {props.name} ({props.total_memory / (1024**3):.1f} GiB VRAM)")

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. 一鍵安裝 FFmpeg 9 與 Python 相依套件
# 1. 安裝官方 FFmpeg 9 PPA
!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg9
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# 2. 取得或更新 SeedVR2-cli 專案
!if [ ! -d "/content/SeedVR2-CLI" ]; then \
    git clone --depth 1 https://github.com/delvilo/SeedVR2-CLI.git /content/SeedVR2-CLI; \
else \
    cd /content/SeedVR2-CLI && git pull; \
fi

# 3. 安裝 Python 相依套件 (移除衝突的 torchvision/torchaudio)
%cd /content/SeedVR2-CLI
!pip install -q safetensors tqdm psutil einops omegaconf diffusers rotary_embedding_torch sageattention gguf
!pip uninstall -q -y torchvision torchaudio

# 4. 建立 Google Drive 工作資料夾
!mkdir -p /content/drive/MyDrive/SeedVR2/input
!mkdir -p /content/drive/MyDrive/SeedVR2/models
!mkdir -p /content/drive/MyDrive/SeedVR2/output

print("✅ 環境與 FFmpeg 9 安裝完成！")


## 🚀 執行影片放大 (依 GPU 硬體選擇專屬最佳作業參數)


In [ ]:
# @title ⚡ 【L4 GPU 專用】極致效能最佳參數 (22GB VRAM · 1080p)
# 特色：
# - 開啟 --keep_on_vram (模型常駐顯存，完全免除 CPU 往返拷貝，提速 2 倍)
# - 全速使用 SageAttention 2 (硬體原生 SM 8.9 Triton 核心加速 1.7x)
# - 使用 7B Q4 模型達到畫質與速度最佳平衡，批次大小 9 提升時序平滑度

import os
os.chdir("/content/SeedVR2-CLI")

INPUT_VIDEO = "/content/drive/MyDrive/SeedVR2/input/video.mp4"
# 若 Drive 中尚未上傳影片，可取消下一行註解直接使用專案自帶範例測試：
# INPUT_VIDEO = "examples/inputs/Mustache_640x360.mp4"

OUTPUT_VIDEO = "/content/drive/MyDrive/SeedVR2/output/l4_upscaled_1080p.mp4"
MODEL_DIR = "/content/drive/MyDrive/SeedVR2/models"

!PYTHONPATH=. python3 inference_cli.py "$INPUT_VIDEO" \
  --output "$OUTPUT_VIDEO" \
  --model_dir "$MODEL_DIR" \
  --dit_model seedvr2_ema_7b-Q4_K_M.gguf \
  --resolution 1080 \
  --batch_size 9 \
  --chunk_size 33 \
  --temporal_overlap 3 \
  --attention_mode sageattn_2 \
  --keep_on_vram \
  --video_backend ffmpeg


In [ ]:
# @title 🛡️ 【T4 GPU 專用】穩定零 OOM 最佳參數 (15GB VRAM · 1080p)
# 特色：
# - 自動使用高速 PyTorch SDPA (避免 Turing SM 7.5 不相容的 Triton 核心)
# - CPU 雙向分相卸載機制 (DiT 與 VAE 交替載入，保證 100% 零 OOM)
# - 輕量 3B Q4 模型，速度快且顯存需求低

import os
os.chdir("/content/SeedVR2-CLI")

INPUT_VIDEO = "/content/drive/MyDrive/SeedVR2/input/video.mp4"
# 若 Drive 中尚未上傳影片，可取消下一行註解直接使用專案自帶範例測試：
# INPUT_VIDEO = "examples/inputs/Mustache_640x360.mp4"

OUTPUT_VIDEO = "/content/drive/MyDrive/SeedVR2/output/t4_upscaled_1080p.mp4"
MODEL_DIR = "/content/drive/MyDrive/SeedVR2/models"

!PYTHONPATH=. python3 inference_cli.py "$INPUT_VIDEO" \
  --output "$OUTPUT_VIDEO" \
  --model_dir "$MODEL_DIR" \
  --dit_model seedvr2_ema_3b-Q4_K_M.gguf \
  --resolution 1080 \
  --batch_size 5 \
  --chunk_size 33 \
  --temporal_overlap 3 \
  --dit_offload_device cpu \
  --vae_offload_device cpu \
  --video_backend ffmpeg


## 🖼️ 單張圖片放大 (可選)


In [ ]:
# @title 🖼️ 單張圖片高畫質放大
import os
os.chdir("/content/SeedVR2-CLI")

INPUT_IMAGE = "examples/inputs/Sadhu_320x478.png"
OUTPUT_IMAGE = "/content/drive/MyDrive/SeedVR2/output/sadhu_upscaled.png"
MODEL_DIR = "/content/drive/MyDrive/SeedVR2/models"

!PYTHONPATH=. python3 inference_cli.py "$INPUT_IMAGE" \
  --output "$OUTPUT_IMAGE" \
  --model_dir "$MODEL_DIR" \
  --dit_model seedvr2_ema_7b-Q4_K_M.gguf \
  --resolution 1080
